In [1]:
# =============================================================================
# SCRIPT 4: Vision Transformer (ViT-Base) — Fungal Keratitis vs. Normal Cornea
# =============================================================================
# Architecture : ViT-B/16 implemented from scratch in Keras (no external lib
#                dependency) — patch size 16, 12 transformer blocks, 12 heads.
#                Falls back gracefully to KerasHub if available.
# Framework    : TensorFlow / Keras  |  Eval: Scikit-Learn, Matplotlib, Seaborn
#
# Install notes (optional speed-up):
#   pip install keras-hub   ← lets you swap in a pretrained ViT easily
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# ✏️  PLUG-AND-PLAY CONFIG
# ──────────────────────────────────────────────────────────────────────────────
import os
import re
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

BASE_DIR        = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
KAGGLE_INPUT    = Path("/kaggle/input")
KAGGLE_WORKING  = Path("/kaggle/working")
OUTPUT_DIR      = KAGGLE_WORKING if KAGGLE_WORKING.exists() else BASE_DIR
DEFAULT_DATASET_PATHS = [
    Path("/kaggle/input/datasets/sanyammahajan/dataset-fuzzy/dataset"),
    Path("/kaggle/input/dataset-fuzzy/dataset"),
]
CLASS_NAMES     = ["FK pic", "normal"]
CLASS_ALIASES   = {
    "FK pic": ["FK pic", "fk pic", "fk", "fungal", "fungal keratitis"],
    "normal": ["normal", "Normal"],
}
MODEL_SAVE_PATH = OUTPUT_DIR / "best_vit_base.keras"
PLOT_SAVE_PATH  = OUTPUT_DIR / "vit_base_evaluation.png"
# ──────────────────────────────────────────────────────────────────────────────

import math
import numpy as np
import tensorflow as tf
import matplotlib

if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── Kaggle GPU setup ──────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    if os.environ.get("USE_MIXED_PRECISION", "1") == "1":
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"\nGPU detected : {[gpu.name for gpu in gpus]}")
    print(f"Precision    : {tf.keras.mixed_precision.global_policy().name}")
else:
    print("\nGPU detected : none")

# ── Hyper-parameters ──────────────────────────────────────────────────────────
IMG_SIZE     = (224, 224)     # spatial input resolution
PATCH_SIZE   = 16             # each patch is PATCH_SIZE × PATCH_SIZE pixels
NUM_PATCHES  = (224 // 16) ** 2  # = 196 patches
D_MODEL      = 768            # transformer embedding dimension (ViT-Base)
NUM_HEADS    = 12             # multi-head attention heads
NUM_LAYERS   = 12             # stacked transformer encoder blocks
MLP_DIM      = 3072           # feed-forward hidden dim (4× D_MODEL)
DROPOUT_RATE = 0.1            # transformer dropout
BATCH_SIZE   = int(os.environ.get("BATCH_SIZE", "16"))
MODEL_TYPE   = os.environ.get("MODEL_TYPE", "efficientnetv2").lower()
EPOCHS       = int(os.environ.get("EPOCHS", "12"))
FINE_TUNE_EPOCHS = int(os.environ.get("FINE_TUNE_EPOCHS", "6"))
FINE_TUNE_LAYERS = int(os.environ.get("FINE_TUNE_LAYERS", "20"))
LR           = float(os.environ.get("LR", "1e-3"))
FINE_TUNE_LR = float(os.environ.get("FINE_TUNE_LR", "1e-5"))
WARMUP_STEPS = 500            # linear LR warm-up to stabilise early training
VAL_SPLIT    = 0.20
GROUP_SPLIT  = os.environ.get("GROUP_SPLIT", "1") == "1"
LABEL_SMOOTHING = float(os.environ.get("LABEL_SMOOTHING", "0.05"))
SHOW_MODEL_SUMMARY = os.environ.get("SHOW_MODEL_SUMMARY", "0") == "1"

# =============================================================================
# 1.  DATA LOADING
# =============================================================================

print("\n[1/6]  Loading datasets …")

IMAGE_EXTS = {".bmp", ".gif", ".jpeg", ".jpg", ".png"}


def normalize_name(name):
    return "".join(ch for ch in name.lower() if ch.isalnum())


def resolve_class_dirs(root):
    if not root.is_dir():
        return None

    child_dirs = [p for p in root.iterdir() if p.is_dir()]
    normalized_children = {normalize_name(p.name): p for p in child_dirs}
    resolved = {}

    for class_name in CLASS_NAMES:
        for alias in CLASS_ALIASES[class_name]:
            match = normalized_children.get(normalize_name(alias))
            if match is not None:
                resolved[class_name] = match
                break

    return resolved if len(resolved) == len(CLASS_NAMES) else None


def find_dataset_root():
    env_path = os.environ.get("DATASET_PATH")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())

    candidates.extend(DEFAULT_DATASET_PATHS)

    if KAGGLE_INPUT.exists():
        candidates.extend([p for p in KAGGLE_INPUT.rglob("*") if p.is_dir()])

    candidates.append(BASE_DIR)

    for candidate in candidates:
        resolved = resolve_class_dirs(candidate)
        if resolved:
            return candidate, resolved

    searched = [str(Path(env_path).expanduser())] if env_path else []
    searched.extend([str(KAGGLE_INPUT), str(BASE_DIR)])
    raise FileNotFoundError(
        "Could not find class folders for FK pic and normal. "
        f"Expected them somewhere under: {searched}. "
        "On Kaggle, add your uploaded dataset to the notebook, or set "
        "DATASET_PATH=/kaggle/input/<dataset-folder>."
    )


DATASET_PATH, CLASS_DIRS = find_dataset_root()
print(f"   Dataset root : {DATASET_PATH}")
print(f"   Output dir   : {OUTPUT_DIR}")


def validate_image(path):
    try:
        raw = tf.io.read_file(str(path))
        tf.io.decode_image(raw, channels=3, expand_animations=False)
        return True
    except tf.errors.InvalidArgumentError:
        return False


def load_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    return image, label


image_paths, labels, skipped = [], [], []
for label, class_name in enumerate(CLASS_NAMES):
    class_dir = CLASS_DIRS[class_name]
    for path in sorted(class_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTS:
            continue
        if validate_image(path):
            image_paths.append(str(path))
            labels.append(label)
        else:
            skipped.append(str(path.relative_to(DATASET_PATH)))

if skipped:
    print(f"   Skipped unreadable images : {skipped}")

if not image_paths:
    raise ValueError(f"No readable images found under {DATASET_PATH}.")

image_paths = np.array(image_paths)
labels = np.array(labels, dtype=np.float32)


def image_group_key(path):
    path = Path(path)
    stem = path.stem.lower()
    stem = re.sub(r"\s*\([^)]*\)", "", stem)
    stem = re.sub(r"\s+(od|os)\b.*", "", stem)
    stem = re.split(r"[_\-\s,]+", stem)[0]
    return f"{path.parent.name}:{stem}"


def stratified_split(labels, val_split, seed):
    rng = np.random.default_rng(seed)
    train_parts, val_parts = [], []
    for class_id in np.unique(labels):
        class_idx = np.where(labels == class_id)[0]
        rng.shuffle(class_idx)
        val_count = max(1, int(round(len(class_idx) * val_split)))
        val_parts.append(class_idx[:val_count])
        train_parts.append(class_idx[val_count:])

    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def stratified_group_split(image_paths, labels, val_split, seed):
    rng = np.random.default_rng(seed)
    train_parts, val_parts = [], []
    group_keys = np.array([image_group_key(path) for path in image_paths])

    for class_id in np.unique(labels):
        class_idx = np.where(labels == class_id)[0]
        class_groups = np.unique(group_keys[class_idx])
        rng.shuffle(class_groups)

        target_val_count = max(1, int(round(len(class_idx) * val_split)))
        selected_val_groups, current_val_count = [], 0
        group_sizes = {
            group: int(np.sum((group_keys == group) & (labels == class_id)))
            for group in class_groups
        }

        for group in class_groups:
            if current_val_count >= target_val_count and selected_val_groups:
                break
            selected_val_groups.append(group)
            current_val_count += group_sizes[group]

        selected_val_groups = set(selected_val_groups)
        class_val_idx = class_idx[np.isin(group_keys[class_idx], list(selected_val_groups))]
        class_train_idx = class_idx[~np.isin(group_keys[class_idx], list(selected_val_groups))]

        val_parts.append(class_val_idx)
        train_parts.append(class_train_idx)

    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx, len(np.unique(group_keys[train_idx])), len(np.unique(group_keys[val_idx]))


if GROUP_SPLIT:
    train_idx, val_idx, train_groups, val_groups = stratified_group_split(
        image_paths, labels, VAL_SPLIT, SEED
    )
    print(f"   Split mode   : grouped by filename stem ({train_groups} train groups, {val_groups} validation groups)")
else:
    train_idx, val_idx = stratified_split(labels, VAL_SPLIT, SEED)
    print("   Split mode   : image-level stratified split")

train_ds = tf.data.Dataset.from_tensor_slices((image_paths[train_idx], labels[train_idx]))
train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)

val_ds = tf.data.Dataset.from_tensor_slices((image_paths[val_idx], labels[val_idx]))
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE)

print(f"Found {len(image_paths)} readable files belonging to {len(CLASS_NAMES)} classes.")
print(f"Using {len(train_idx)} files for training.")
print(f"Using {len(val_idx)} files for validation.")
print(f"   Classes : {CLASS_NAMES}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

# =============================================================================
# 2.  CLASS-IMBALANCE HANDLING
# =============================================================================

print("\n[2/6]  Computing class weights …")

n_classes     = len(CLASS_NAMES)
counts        = [int(np.sum(labels == i)) for i in range(n_classes)]
total_samples = sum(counts)
class_weights = {i: total_samples / (n_classes * counts[i]) for i in range(n_classes)}
print(f"   Sample counts : { {CLASS_NAMES[i]: counts[i] for i in range(n_classes)} }")
print(f"   Class weights : { {CLASS_NAMES[i]: round(class_weights[i], 4) for i in range(n_classes)} }")

# =============================================================================
# 3.  DATA AUGMENTATION
# =============================================================================

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        tf.keras.layers.RandomRotation(0.20, seed=SEED),
        tf.keras.layers.RandomTranslation(0.08, 0.08, seed=SEED),
        tf.keras.layers.RandomZoom(0.20, seed=SEED),
        tf.keras.layers.RandomContrast(0.15, seed=SEED),
    ],
    name="augmentation",
)

# =============================================================================
# 4.  VISION TRANSFORMER — BUILDING BLOCKS
# =============================================================================
# The ViT splits an image into a grid of fixed-size patches, linearly embeds
# each patch, prepends a learnable [CLS] token, adds positional embeddings,
# and feeds the sequence through standard Transformer Encoder blocks.
# The [CLS] token's final representation is fed to the classification head.

# ── 4a. Patch Extraction & Embedding ─────────────────────────────────────────
class PatchEmbedding(tf.keras.layers.Layer):
    """
    Splits a (H, W, C) image into non-overlapping patches of size P×P,
    flattens each patch to a vector, and linearly projects to D_MODEL dims.
    Also prepends the learnable [CLS] token and adds learnable 1D positional
    embeddings (one per patch + one for CLS).
    """
    def __init__(self, patch_size, num_patches, d_model, **kwargs):
        super().__init__(**kwargs)
        self.patch_size  = patch_size
        self.num_patches = num_patches
        self.d_model     = d_model

        # Linear projection of flattened patches
        self.projection = tf.keras.layers.Dense(d_model)

        # Learnable [CLS] token — one vector shared across the batch
        self.cls_token = self.add_weight(
            shape=(1, 1, d_model),
            initializer="zeros",
            trainable=True,
            name="cls_token",
        )
        # Learnable positional embeddings: (1, num_patches + 1, d_model)
        self.pos_embedding = self.add_weight(
            shape=(1, num_patches + 1, d_model),
            initializer="random_normal",
            trainable=True,
            name="pos_embedding",
        )

    def call(self, images):
        batch_size = tf.shape(images)[0]
        p = self.patch_size

        # Extract patches: (B, H/P, W/P, P*P*C) via reshape
        # Step 1 — reshape spatially into patch grid
        x = tf.reshape(
            images,
            (batch_size,
             IMG_SIZE[0] // p, p,
             IMG_SIZE[1] // p, p,
             3)
        )
        # Step 2 — rearrange dims: (B, n_h, n_w, p, p, C)
        x = tf.transpose(x, perm=[0, 1, 3, 2, 4, 5])
        # Step 3 — flatten each patch: (B, num_patches, patch_pixels)
        x = tf.reshape(x, (batch_size, self.num_patches, p * p * 3))

        # Project each patch vector to D_MODEL
        x = self.projection(x)   # (B, num_patches, D_MODEL)

        # Prepend CLS token (broadcast over batch)
        cls_tokens = tf.broadcast_to(self.cls_token, (batch_size, 1, self.d_model))
        x = tf.concat([cls_tokens, x], axis=1)  # (B, num_patches+1, D_MODEL)

        # Add positional embeddings (learned, not sinusoidal)
        x = x + self.pos_embedding   # (B, num_patches+1, D_MODEL)
        return x


# ── 4b. Multi-Head Self-Attention Block ───────────────────────────────────────
class TransformerEncoderBlock(tf.keras.layers.Layer):
    """
    Standard Pre-LN Transformer Encoder Block:
        x → LN → MHSA → residual → LN → MLP → residual
    Pre-normalisation (LN before the sub-layer) gives more stable ViT training.
    """
    def __init__(self, d_model, num_heads, mlp_dim, dropout_rate, **kwargs):
        super().__init__(**kwargs)

        # Multi-Head Self-Attention
        self.attn = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,   # per-head dimension
            dropout=dropout_rate,
        )
        # Position-wise Feed-Forward MLP
        self.mlp = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(mlp_dim, activation="gelu"),  # GELU → ViT default
                tf.keras.layers.Dropout(dropout_rate),
                tf.keras.layers.Dense(d_model),
                tf.keras.layers.Dropout(dropout_rate),
            ]
        )
        # Layer Normalisation (applied before each sub-layer — "Pre-LN")
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop  = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        # Self-attention residual branch
        attn_out = self.attn(
            self.norm1(x), self.norm1(x), training=training
        )
        x = x + self.drop(attn_out, training=training)   # residual connection

        # MLP residual branch
        mlp_out = self.mlp(self.norm2(x), training=training)
        x = x + mlp_out                                   # residual connection
        return x


# =============================================================================
# 5.  MODEL DEFINITION  —  Full ViT-Base
# =============================================================================

print(f"\n[3/6]  Building model: {MODEL_TYPE} …")

def build_vit(
    img_size    = IMG_SIZE,
    patch_size  = PATCH_SIZE,
    num_patches = NUM_PATCHES,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    num_layers  = NUM_LAYERS,
    mlp_dim     = MLP_DIM,
    dropout     = DROPOUT_RATE,
):
    inputs = tf.keras.Input(shape=(*img_size, 3), name="input_image")

    # Augment and normalise pixels to [0, 1]
    x = augmentation(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 255)(x)

    # Patch embedding: image → sequence of patch vectors + CLS + pos encoding
    x = PatchEmbedding(patch_size, num_patches, d_model, name="patch_embed")(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    # Stack of Transformer Encoder blocks
    for block_idx in range(num_layers):
        x = TransformerEncoderBlock(
            d_model, num_heads, mlp_dim, dropout,
            name=f"transformer_block_{block_idx}"
        )(x)

    # Final LayerNorm over the full sequence
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)

    # Extract the [CLS] token representation (index 0 along sequence axis)
    cls_out = x[:, 0, :]     # shape (B, D_MODEL)

    # MLP classification head
    x       = tf.keras.layers.Dense(256, activation="gelu")(cls_out)
    x       = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="prediction",
    )(x)

    return tf.keras.Model(inputs, outputs, name="ViT_Base")


def build_efficientnetv2():
    weights = os.environ.get("PRETRAINED_WEIGHTS", "imagenet")
    weights = None if weights.lower() in {"none", "false", "0"} else weights
    app_model = getattr(tf.keras.applications, "EfficientNetV2B0", None)
    model_name = "EfficientNetV2B0"
    model_kwargs = {"include_preprocessing": True}
    if app_model is None:
        app_model = tf.keras.applications.EfficientNetB0
        model_name = "EfficientNetB0"
        model_kwargs = {}

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name="input_image")
    x = augmentation(inputs)

    try:
        base_model = app_model(
            include_top=False,
            weights=weights,
            input_tensor=x,
            **model_kwargs,
        )
    except Exception as exc:
        if weights is None:
            raise
        print(f"   Could not load ImageNet weights ({exc}). Falling back to random weights.")
        base_model = app_model(
            include_top=False,
            weights=None,
            input_tensor=x,
            **model_kwargs,
        )

    base_model.trainable = False
    x = base_model.output
    x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.50)(x)
    x = tf.keras.layers.Dense(
        128,
        activation="swish",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name="prediction",
    )(x)

    model = tf.keras.Model(inputs, outputs, name=f"{model_name}_FK")
    return model, base_model


def build_model():
    if MODEL_TYPE in {"vit", "vit-base", "vit_base"}:
        return build_vit(), None
    if MODEL_TYPE in {"efficientnet", "efficientnetv2", "efficientnetv2b0"}:
        return build_efficientnetv2()
    raise ValueError(
        f"Unsupported MODEL_TYPE={MODEL_TYPE!r}. "
        "Use 'efficientnetv2' or 'vit'."
    )


model, base_model = build_model()
if SHOW_MODEL_SUMMARY:
    model.summary()
else:
    print(
        f"   Params       : {model.count_params():,} "
        "(set SHOW_MODEL_SUMMARY=1 to print full layer table)"
    )

# =============================================================================
# 6.  LEARNING RATE SCHEDULE  —  Linear Warm-Up + Cosine Decay
# =============================================================================
# ViTs are sensitive to the early training phase; a warm-up prevents large
# gradient updates before the attention weights have settled.

class WarmUpCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warm-up for `warmup_steps`, then cosine decay to `min_lr`."""
    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.initial_lr   = initial_lr
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / self.warmup_steps)
        cosine_decay = self.min_lr + 0.5 * (self.initial_lr - self.min_lr) * (
            1 + tf.math.cos(
                math.pi * (step - self.warmup_steps)
                / (self.total_steps - self.warmup_steps)
            )
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_decay)

    def get_config(self):
        return {
            "initial_lr":   self.initial_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps":  self.total_steps,
            "min_lr":       self.min_lr,
        }

# Estimate total training steps (upper bound using max EPOCHS)
steps_per_epoch = math.ceil(
    (total_samples * (1 - VAL_SPLIT)) / BATCH_SIZE
)
total_steps = EPOCHS * steps_per_epoch

lr_schedule = WarmUpCosineDecay(
    initial_lr=LR,
    warmup_steps=WARMUP_STEPS,
    total_steps=total_steps,
    min_lr=1e-7,
)

# =============================================================================
# 7.  COMPILATION & CALLBACKS
# =============================================================================

print("\n[4/6]  Compiling model …")

def make_optimizer(learning_rate):
    try:
        return tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=1e-4,
            clipnorm=1.0,
        )
    except AttributeError:
        return tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0)


def compile_model(learning_rate):
    model.compile(
        optimizer=make_optimizer(learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )


compile_model(lr_schedule if MODEL_TYPE in {"vit", "vit-base", "vit_base"} else LR)

def make_callbacks():
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
    ]
    if MODEL_TYPE not in {"vit", "vit-base", "vit_base"}:
        callbacks.insert(
            1,
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.3,
                patience=2,
                min_lr=1e-7,
                verbose=1,
            ),
        )
    return callbacks

# =============================================================================
# 8.  TRAINING
# =============================================================================

print("\n[5/6]  Training model …")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=make_callbacks(),
    verbose=1,
)

if base_model is not None and FINE_TUNE_EPOCHS > 0:
    print(f"\n[5/6]  Fine-tuning last {FINE_TUNE_LAYERS} base-model layers …")
    base_model.trainable = True
    for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    compile_model(FINE_TUNE_LR)
    fine_tune_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS + FINE_TUNE_EPOCHS,
        initial_epoch=EPOCHS,
        class_weight=class_weights,
        callbacks=make_callbacks(),
        verbose=1,
    )

    for key, values in fine_tune_history.history.items():
        history.history.setdefault(key, []).extend(values)

# =============================================================================
# 9.  EVALUATION & VISUALISATION
# =============================================================================

print("\n[6/6]  Evaluating and plotting …")

y_true, y_prob = [], []
for images, labels in val_ds:
    probs = model(images, training=False).numpy().flatten()
    y_prob.extend(probs)
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print("\n── Classification Report ──────────────────────────────────────")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Vision Transformer (ViT-Base) — Evaluation Dashboard",
             fontsize=16, fontweight="bold")

# ── Panel 1: Accuracy ─────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(history.history["accuracy"],     label="Train Acc", lw=2)
ax.plot(history.history["val_accuracy"], label="Val Acc",   lw=2, ls="--")
ax.set_title("Accuracy over Epochs"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(alpha=0.3)

# ── Panel 2: Loss ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(history.history["loss"],     label="Train Loss", lw=2)
ax.plot(history.history["val_loss"], label="Val Loss",   lw=2, ls="--")
ax.set_title("Loss over Epochs"); ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
ax.legend(); ax.grid(alpha=0.3)

# ── Panel 3: ROC ──────────────────────────────────────────────────────────────
ax = axes[0, 2]
fpr, tpr, _ = roc_curve(y_true, y_prob)
auroc = auc(fpr, tpr)
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUROC = {auroc:.4f}")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("ROC Curve"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)

# ── Panel 4: PR Curve ─────────────────────────────────────────────────────────
ax = axes[1, 0]
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
auprc = average_precision_score(y_true, y_prob)
ax.plot(recall_vals, precision_vals, color="darkorange", lw=2, label=f"AUPRC = {auprc:.4f}")
ax.set_title("Precision-Recall Curve"); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)

# ── Panel 5: Confusion Matrix ─────────────────────────────────────────────────
ax = axes[1, 1]
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label")

# ── Panel 6: Per-Class Bar Chart ──────────────────────────────────────────────
ax = axes[1, 2]
report  = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
grp     = CLASS_NAMES + ["macro avg"]
metrics = ["precision", "recall", "f1-score"]
data    = np.array([[report[c][m] for m in metrics] for c in grp])
x       = np.arange(len(grp)); w = 0.25
for i, m in enumerate(metrics):
    ax.bar(x + i * w, data[:, i], w, label=m.capitalize())
ax.set_xticks(x + w); ax.set_xticklabels(grp, rotation=15, fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score"); ax.set_title("Per-Class Metrics")
ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_SAVE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅  Saved plot to  {PLOT_SAVE_PATH}")
print(f"   Saved model to {MODEL_SAVE_PATH}")
print(f"   AUROC : {auroc:.4f}   |   AUPRC : {auprc:.4f}")


2026-05-10 17:41:43.107195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778434903.457223      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778434903.555611      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778434904.503210      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778434904.503263      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778434904.503266      23 computation_placer.cc:177] computation placer alr


GPU detected : ['/physical_device:GPU:0']
Precision    : mixed_float16

[1/6]  Loading datasets …
   Dataset root : /kaggle/input/datasets/sanyammahajan/dataset-fuzzy/dataset
   Output dir   : /kaggle/working


I0000 00:00:1778434945.163839      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


   Skipped unreadable images : ['normal/75_0.jpg', 'normal/75_1.jpg', 'normal/75_2.jpg']
   Split mode   : grouped by filename stem (100 train groups, 52 validation groups)
Found 1396 readable files belonging to 2 classes.
Using 1113 files for training.
Using 283 files for validation.
   Classes : ['FK pic', 'normal']

[2/6]  Computing class weights …
   Sample counts : {'FK pic': 863, 'normal': 533}
   Class weights : {'FK pic': 0.8088, 'normal': 1.3096}

[3/6]  Building model: efficientnetv2 …
24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
   Params       : 6,089,041 (set SHOW_MODEL_SUMMARY=1 to print full layer table)

[4/6]  Compiling model …

[5/6]  Training model …
Epoch 1/12


E0000 00:00:1778434972.646696      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetV2B0_FK_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1778434977.052977      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


69/70 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.8572 - auc: 0.9337 - loss: 0.4058 - precision: 0.7759 - recall: 0.8667
Epoch 1: val_loss improved from inf to 0.29902, saving model to /kaggle/working/best_vit_base.keras
70/70 ━━━━━━━━━━━━━━━━━━━━ 44s 258ms/step - accuracy: 0.8592 - auc: 0.9351 - loss: 0.4030 - precision: 0.7788 - recall: 0.8690 - val_accuracy: 0.9823 - val_auc: 0.9916 - val_loss: 0.2990 - val_precision: 0.9725 - val_recall: 0.9815 - learning_rate: 0.0010
Epoch 2/12
69/70 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9681 - auc: 0.9929 - loss: 0.2428 - precision: 0.9505 - recall: 0.9687
Epoch 2: val_loss improved from 0.29902 to 0.26035, saving model to /kaggle/working/best_vit_base.keras
70/70 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.9681 - auc: 0.9930 - loss: 0.2427 - precision: 0.9504 - recall: 0.9687 - val_accuracy: 0.9788 - val_auc: 0.9955 - val_loss: 0.2604 - val_precision: 0.9636 - val_recall: 0.9815 - learning_rate: 0.0010
Epoch 3/12
69/70 ━━━━━━━

E0000 00:00:1778435050.571539      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetV2B0_FK_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


69/70 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.9845 - auc: 0.9993 - loss: 0.1985 - precision: 0.9825 - recall: 0.9771
Epoch 13: val_loss improved from inf to 0.19304, saving model to /kaggle/working/best_vit_base.keras
70/70 ━━━━━━━━━━━━━━━━━━━━ 27s 143ms/step - accuracy: 0.9844 - auc: 0.9993 - loss: 0.1984 - precision: 0.9823 - recall: 0.9772 - val_accuracy: 0.9788 - val_auc: 0.9988 - val_loss: 0.1930 - val_precision: 0.9904 - val_recall: 0.9537 - learning_rate: 1.0000e-05
Epoch 14/18
69/70 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9715 - auc: 0.9955 - loss: 0.2211 - precision: 0.9691 - recall: 0.9574
Epoch 14: val_loss improved from 0.19304 to 0.19293, saving model to /kaggle/working/best_vit_base.keras
70/70 ━━━━━━━━━━━━━━━━━━━━ 4s 56ms/step - accuracy: 0.9717 - auc: 0.9956 - loss: 0.2208 - precision: 0.9692 - recall: 0.9578 - val_accuracy: 0.9788 - val_auc: 0.9989 - val_loss: 0.1929 - val_precision: 0.9904 - val_recall: 0.9537 - learning_rate: 1.0000e-05
Epoch 15/18
69